# Action Recommendation Notebook

This notebook focuses on operational action recommendations from delay risk scores.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [ ]:
file_path = "/home/pratyush_device/Documents/college/AIML lab/Project/customer_cleaned.csv"
df = pd.read_csv(file_path)
print("Loaded shape:", df.shape)
df.head()

In [ ]:
target_col = "lead_time_days"
if target_col not in df.columns:
    raise ValueError("lead_time_days column not found in customer_cleaned.csv")

delay_threshold = df[target_col].quantile(0.75)
df["delay"] = (df[target_col] > delay_threshold).astype(int)

X_raw = df.drop(columns=[target_col, "delay"])
X = pd.get_dummies(X_raw, drop_first=True)
y = df["delay"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)

print("Delay threshold:", delay_threshold)
print("Model trained for recommendation scoring.")

In [ ]:
def risk_level(prob):
    if prob >= 0.70:
        return "High"
    if prob >= 0.40:
        return "Medium"
    return "Low"


def action_plan(level):
    if level == "High":
        return {
            "Action": "Expedite shipment and escalate supplier",
            "Priority": "P1",
            "Owner": "Supply Chain Manager",
            "SLA_Hours": 12
        }
    if level == "Medium":
        return {
            "Action": "Monitor order and confirm dispatch timeline",
            "Priority": "P2",
            "Owner": "Operations Analyst",
            "SLA_Hours": 24
        }
    return {
        "Action": "No immediate action; keep routine monitoring",
        "Priority": "P3",
        "Owner": "Auto-monitoring",
        "SLA_Hours": 72
    }

In [ ]:
# Build recommendations for all rows
delay_prob = clf.predict_proba(X)[:, 1]
recommendations = X_raw.copy()
recommendations["Delay_Probability"] = delay_prob
recommendations["Risk_Level"] = recommendations["Delay_Probability"].apply(risk_level)

action_df = recommendations["Risk_Level"].apply(action_plan).apply(pd.Series)
recommendations = pd.concat([recommendations, action_df], axis=1)

print(recommendations["Risk_Level"].value_counts())
recommendations.head(10)

In [ ]:
# Action queue: show highest-risk orders first
queue_cols = ["Delay_Probability", "Risk_Level", "Priority", "Owner", "SLA_Hours", "Action"]
action_queue = recommendations.sort_values(by="Delay_Probability", ascending=False).reset_index(drop=True)
action_queue[queue_cols].head(15)

In [ ]:
# Single-order recommendation helper
def recommend_for_order(input_row):
    input_encoded = pd.get_dummies(input_row, drop_first=True)
    input_encoded = input_encoded.reindex(columns=X.columns, fill_value=0)
    prob = float(clf.predict_proba(input_encoded)[0][1])
    level = risk_level(prob)
    plan = action_plan(level)
    return {
        "Delay_Probability": prob,
        "Risk_Level": level,
        **plan
    }

sample = X_raw.iloc[[0]]
recommend_for_order(sample)

In [ ]:
output_path = "/home/pratyush_device/Documents/college/AIML lab/Project/action_recommendations.csv"
action_queue.to_csv(output_path, index=False)
print("Saved action recommendation file to:", output_path)